# Bilateral Data Fetch: Hemispheric Lateralization Analysis

## Motivation: Why Bilateral Data?

### The Problem
Most neuroscience analyses treat hemispheres independently, ignoring the fact that **animals have two brains that must coordinate**. This notebook fetches **bilateral recordings** (simultaneous neural activity from left and right hemispheres) to directly measure hemispheric asymmetries and coordination.

### Why This Matters

1. **Hemispheric Specialization**: Not all regions are equally lateralized. Motor cortex may be specialized (dominant hand control), while other areas share the load.

2. **Computational Independence**: How much does the right hemisphere do independently vs. copying the left? Bilateral data reveals this division of labor.

3. **Behavioral Coordination**: Movement is fundamentally bilateral—reaching with one arm still requires coordinated postural control on the other side. Bilateral neural recordings show how this is orchestrated.

4. **Choice Encoding Asymmetry**: Does one hemisphere "decide" while the other executes? Or do both hemispheres encode choice with different latencies/magnitudes? Only bilateral data can answer this.

### Key Questions We Address

- **Q1**: Which regions show unilateral vs. bilateral choice encoding?
- **Q2**: Are hemispheres synchronized or independent? (measured via CCA coupling)?

### Data Strategy

We use **paired bilateral insertions** from the International Brain Laboratory (IBL) dataset:
- Simultaneous recordings from homologous brain regions across hemispheres
- Multiple animals & sessions for statistical robustness
- Aligned to behavior (choice, movement onset, outcome)

This notebook **fetches and organizes** this bilateral data as the foundation for downstream CCA analysis (measuring coupling strength) and choice encoding analysis (measuring lateralization of decision signals).


In [ ]:
! pip install ONE-api
! pip install ibllib
!pip install elephant
!pip install numba

In [ ]:
import numpy as np
import pandas as pd
from one.api import ONE
from sklearn.decomposition import PCA
from brainbox.io.one import SpikeSortingLoader
from iblatlas.atlas import AllenAtlas
import matplotlib.pyplot as plt
#plotting and analysis libs
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.cm as cm

In [ ]:
from one.api import ONE
ONE.setup(base_url='https://openalyx.internationalbrainlab.org', silent=True)
one = ONE(password='international')

In [ ]:
one.load_cache(tag='Brainwidemap')  # release tag shown in docs :contentReference[oaicite:3]{index=3}

# How many insertions exist in this release with spikes?
pids_all = one.search_insertions(project='brainwide', datasets='spikes.times.npy')
print("Insertions with spikes.times:", len(pids_all))

In [ ]:
from iblatlas.atlas import AllenAtlas
ba = AllenAtlas()

In [ ]:
# Bilateral targets from the BWM release doc (mm, bregma-style)
BILAT_TARGETS = [
    {"name": "AP-2.0_LR2.4_repeated", "ap_mm": -2.0, "lr_abs_mm": 2.4},
    {"name": "AP-4.0_LR1.8",          "ap_mm": -4.0, "lr_abs_mm": 1.8},
    {"name": "AP+2.5_LR1.3",          "ap_mm":  2.5, "lr_abs_mm": 1.3},
]

# Prefer "true" tracks, but allow Planned as fallback to locate candidates
PROV_PRIORITY = [
    "Ephys aligned histology track",
    "Histology track",
    "Micro-manipulator",
    "Planned",
]

def um_to_mm(v):
    """Your sample shows coords are microns (µm), so convert to mm."""
    return float(v) / 1000.0

def get_traj_by_priority(one, pid):
    """Return best trajectory dict + provenance based on priority list."""
    for prov in PROV_PRIORITY:
        trs = one.alyx.rest("trajectories", "list", probe_insertion=pid, provenance=prov, no_cache=True)
        if trs:
            t = trs[0]
            # require coordinates
            if t.get("x") is not None and t.get("y") is not None and t.get("z") is not None:
                return t, prov
    return None, None

def assign_target(ap_mm, lr_mm, tol_ap=0.8, tol_lr=0.8):
    """Match insertion entry point (AP, |LR|) to one of the 3 bilateral targets."""
    best = None
    best_score = np.inf
    for t in BILAT_TARGETS:
        dap = abs(ap_mm - t["ap_mm"])
        dlr = abs(abs(lr_mm) - t["lr_abs_mm"])
        if dap <= tol_ap and dlr <= tol_lr:
            score = dap + dlr
            if score < best_score:
                best_score = score
                best = t["name"]
    return best

def find_bilateral_target_sessions(one, tol_ap=1.0, tol_lr=1.0):
    # All insertions with spikes
    pids = one.search_insertions(project="brainwide", datasets="spikes.times.npy")
    try:
        pids = pids[0:len(pids)]  # safe slicing
    except TypeError:
        pids = list(pids)

    rows = []
    for pid in pids:
        ins = one.alyx.rest("insertions", "list", id=pid, no_cache=True)
        if not ins:
            continue
        eid = ins[0]["session"]

        tr, prov = get_traj_by_priority(one, pid)
        if tr is None:
            continue

        # Coordinates (µm -> mm)
        x_mm = um_to_mm(tr["x"])  # ML (LR)
        y_mm = um_to_mm(tr["y"])  # AP
        z_mm = um_to_mm(tr["z"])  # DV (entry ~0)

        target = assign_target(ap_mm=y_mm, lr_mm=x_mm, tol_ap=tol_ap, tol_lr=tol_lr)
        if target is None:
            continue

        hemi = "L" if x_mm < 0 else "R"  # midline is ~0 in this coord system
        rows.append({
            "eid": eid, "pid": pid, "hemi": hemi,
            "target": target, "prov": prov,
            "x_mm": x_mm, "y_mm": y_mm, "z_mm": z_mm
        })

    hits_df = pd.DataFrame(rows)
    if hits_df.empty:
        return hits_df, pd.DataFrame()

    # Find sessions that have both hemispheres for the SAME target
    pairs = []
    for (eid, target), g in hits_df.groupby(["eid", "target"]):
        hemis = set(g["hemi"].values)
        if "L" in hemis and "R" in hemis:
            pid_L = g.loc[g["hemi"] == "L", "pid"].iloc[0]
            pid_R = g.loc[g["hemi"] == "R", "pid"].iloc[0]
            pairs.append({
                "eid": eid, "target": target,
                "pid_left": pid_L, "pid_right": pid_R,
                "prov_left": g.loc[g["hemi"] == "L", "prov"].iloc[0],
                "prov_right": g.loc[g["hemi"] == "R", "prov"].iloc[0],
                "n_hits": len(g),
            })

    pairs_df = pd.DataFrame(pairs).sort_values(["target", "eid"]).reset_index(drop=True)
    return hits_df, pairs_df

# --- Run ---
hits_df, bilat_pairs_df = find_bilateral_target_sessions(one, tol_ap=1.0, tol_lr=1.0)

print("Insertions near any bilateral target (hits):", len(hits_df))
print("Sessions with bilateral L+R at same target:", len(bilat_pairs_df))
display(bilat_pairs_df.head(20))